# Count Graph
- Reload optimal composite for each batch / balance (6 combinations in total)
- Generate combined ranking for the subnetworks based on the specifications of the chosen composite (chosen centralities for high and low + applied weight) (both weighted and with threshold)
- Calculate correlation (2 per network, weighted and threshold) against each ground truth (4 corr. Scores)
- For each group of networks (unbalanced+50 cutoff, unbalanced+100 cutoff…etc…), generate graph counting the number of overall best-correlation of individual centralities to gt, including the count of the times that the composite metric outperforms the individual centralities in that network

In [18]:
import pandas as pd
import os
import json
from collections import Counter


In [19]:
def analyze_best_centralities(base_path: str = 'results/merged-subarticles-edges/importance-merged'):
    """
    Analyze best centralities and composite rankings across all analysis_results.json files.
    Tracks best weight and threshold combinations for each network.
    """    
    
    # Initialize counters for basic centralities
    best_high_counts = Counter()
    best_low_counts = Counter()
    best_overall_counts = Counter()
    
    # Track best composite configurations per network
    class CompositeResult:
        def __init__(self, optimized_for, weight_or_threshold, corr_importance, corr_doctypebranch, high_centrality, low_centrality):
            self.optimized_for = optimized_for
            self.value = weight_or_threshold
            self.corr_importance = corr_importance
            self.corr_doctypebranch = corr_doctypebranch
            self.avg_corr = (corr_importance + corr_doctypebranch) / 2
            self.high_centrality = high_centrality
            self.low_centrality = low_centrality
    
    # Dictionary to store best results for each network
    network_results = {}
    
    for root, dirs, files in os.walk(base_path):
        if 'comparisons' in root:
            continue
            
        if 'analysis_results.json' in files:
            try:
                network_name = os.path.basename(root)
                with open(os.path.join(root, 'analysis_results.json'), 'r') as f:
                    results = json.load(f)
                
                # Extract best basic centralities
                for ground_truth, analysis in results.get('ground_truth_analysis', {}).items():
                    best_centralities = analysis.get('best_centralities', {})
                    if 'high' in best_centralities:
                        best_high_counts[best_centralities['high']] += 1
                    if 'low' in best_centralities:
                        best_low_counts[best_centralities['low']] += 1
                    if 'overall' in best_centralities:
                        best_overall_counts[best_centralities['overall']] += 1
                
                # Track weight and threshold results for this network
                weight_results = []
                threshold_results = []
                
                ground_truth_analysis = results.get('ground_truth_analysis', {})
                
                # For each ground truth that was used for optimization
                for optimized_for, analysis in ground_truth_analysis.items():
                    weight = analysis.get('weight')
                    threshold = analysis.get('threshold')
                    correlations = analysis.get('correlations', {})
                    best_centralities = analysis.get('best_centralities', {})
                    
                    # Get the centralities used for this ground truth
                    high_centrality = best_centralities.get('high')
                    low_centrality = best_centralities.get('low')
                    
                    # Find correlations for both methods
                    weight_importance_corr = None
                    weight_doctypebranch_corr = None
                    threshold_importance_corr = None
                    threshold_doctypebranch_corr = None
                    
                    for pair, corr in correlations.items():
                        pair_str = str(pair)
                        if f"composite_ranking_{optimized_for}_weight_composite_ranking" in pair_str:
                            if "'importance'" in pair_str:
                                weight_importance_corr = corr
                            elif "'doctypebranch'" in pair_str:
                                weight_doctypebranch_corr = corr
                        elif f"composite_ranking_{optimized_for}_threshold_composite_ranking" in pair_str:
                            if "'importance'" in pair_str:
                                threshold_importance_corr = corr
                            elif "'doctypebranch'" in pair_str:
                                threshold_doctypebranch_corr = corr
                    
                    # Add results if we found both correlations
                    if weight_importance_corr is not None and weight_doctypebranch_corr is not None:
                        weight_results.append(
                            CompositeResult(
                                optimized_for,
                                weight,
                                weight_importance_corr,
                                weight_doctypebranch_corr,
                                high_centrality,
                                low_centrality
                            )
                        )
                    
                    if threshold_importance_corr is not None and threshold_doctypebranch_corr is not None:
                        threshold_results.append(
                            CompositeResult(
                                optimized_for,
                                threshold,
                                threshold_importance_corr,
                                threshold_doctypebranch_corr,
                                high_centrality,
                                low_centrality
                            )
                        )
                
                # Find best weight and threshold configurations for this network
                if weight_results:
                    best_weight = max(weight_results, key=lambda x: x.avg_corr)
                else:
                    best_weight = None
                    
                if threshold_results:
                    best_threshold = max(threshold_results, key=lambda x: x.avg_corr)
                else:
                    best_threshold = None
                
                # Store results for this network
                network_results[network_name] = {
                    'weight': best_weight,
                    'threshold': best_threshold
                }
                
                # Print results for this network
                print(f"\nNetwork: {network_name}")
                if best_weight:
                    print(f"Best Weight Configuration:")
                    print(f"  Optimized for: {best_weight.optimized_for}")
                    print(f"  Weight: {best_weight.value}")
                    print(f"  High centrality: {best_weight.high_centrality}")
                    print(f"  Low centrality: {best_weight.low_centrality}")
                    print(f"  Correlation with importance: {best_weight.corr_importance:.4f}")
                    print(f"  Correlation with doctypebranch: {best_weight.corr_doctypebranch:.4f}")
                    print(f"  Average correlation: {best_weight.avg_corr:.4f}")
                
                if best_threshold:
                    print(f"Best Threshold Configuration:")
                    print(f"  Optimized for: {best_threshold.optimized_for}")
                    print(f"  Threshold: {best_threshold.value}")
                    print(f"  High centrality: {best_threshold.high_centrality}")
                    print(f"  Low centrality: {best_threshold.low_centrality}")
                    print(f"  Correlation with importance: {best_threshold.corr_importance:.4f}")
                    print(f"  Correlation with doctypebranch: {best_threshold.corr_doctypebranch:.4f}")
                    print(f"  Average correlation: {best_threshold.avg_corr:.4f}")
                
            except Exception as e:
                print(f"Error processing {root}: {str(e)}")
                continue
    
    # Create summary DataFrame    
    summary = pd.DataFrame({
        'High Centrality': pd.Series(best_high_counts),
        'Low Centrality': pd.Series(best_low_counts),
        'Overall Best': pd.Series(best_overall_counts)
    })
    
    # Save summary to CSV
    output_path = os.path.join(base_path, 'centrality_summary.csv')
    summary.to_csv(output_path)
    
    # Print final summary
    print("\n=== Final Summary ===")
    print("\nBest High Centrality Counts:")
    for centrality, count in best_high_counts.most_common():
        print(f"{centrality}: {count}")
        
    print("\nBest Low Centrality Counts:")
    for centrality, count in best_low_counts.most_common():
        print(f"{centrality}: {count}")
        
    print("\nBest Overall Centrality Counts:")
    for centrality, count in best_overall_counts.most_common():
        print(f"{centrality}: {count}")
    
    return {
        'best_high_counts': best_high_counts,
        'best_low_counts': best_low_counts,
        'best_overall_counts': best_overall_counts,
        'network_results': network_results,
        'summary_df': summary
    }

In [20]:
results = analyze_best_centralities('results/merged-subarticles-edges/importance-merged')
print(results)


Network: full-balanced-doctypebranch
Best Weight Configuration:
  Optimized for: importance
  Weight: 1.0
  High centrality: in_degree_centrality
  Low centrality: relative_in_degree_centrality
  Correlation with importance: 0.7310
  Correlation with doctypebranch: 0.7074
  Average correlation: 0.7192
Best Threshold Configuration:
  Optimized for: importance
  Threshold: 1
  High centrality: in_degree_centrality
  Low centrality: relative_in_degree_centrality
  Correlation with importance: 0.7310
  Correlation with doctypebranch: 0.7074
  Average correlation: 0.7192

Network: full-balanced-importance
Best Weight Configuration:
  Optimized for: importance
  Weight: 1.0
  High centrality: in_degree_centrality
  Low centrality: relative_in_degree_centrality
  Correlation with importance: 0.5477
  Correlation with doctypebranch: 0.4451
  Average correlation: 0.4964
Best Threshold Configuration:
  Optimized for: importance
  Threshold: 1
  High centrality: in_degree_centrality
  Low centra